<a href="https://colab.research.google.com/github/gro5-AberUni/Sahel_Inundation_Mapping/blob/main/tropWetUnmix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-gregoryoakessentone')

def bitwiseExtract(value, fromBit, toBit):
  if toBit == None:
    toBit = fromBit
  maskSize = ee.Number(1).add(toBit).subtract(fromBit)
  mask = ee.Number(1).leftShift(maskSize).subtract(1)
  return value.rightShift(fromBit).bitwiseAnd(mask)

def cloudMaskLSOLI(image):
  cloudShadowBitMask = (1 << 4);
  cloudsBitMask = (1 << 3);
  cirrusBitMask = (1<<2)
  snowBitMask = (1<<5)
  qa = image.select('QA_PIXEL');
  ra = image.select('QA_RADSAT')

  mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0)).And(qa.bitwiseAnd(cirrusBitMask).eq(0)).Or(ra.bitwiseAnd(1<<1))
  anySaturated = bitwiseExtract(ra, 1, 7).eq(0)
  snowMask = qa.bitwiseAnd(snowBitMask).eq(0)

  opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2);
  thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0);
  correctedImg =  image.addBands(opticalBands, None, True).addBands(thermalBands, None, True);

  bandNames = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'];
  maskedImage = correctedImg.updateMask(mask);
  maskedImgSnow = maskedImage.updateMask(snowMask)
  maskedImgSnowSat = maskedImgSnow.updateMask(anySaturated)
  maskedImageSelectedBands = maskedImgSnow.select(bandNames);
  maskedImageSelectedBands = maskedImageSelectedBands.rename('b','g','r','nir','swir1','swir2')

  maskedImageSelectedBandsShort = maskedImageSelectedBands.multiply(10000)

  blue = maskedImageSelectedBandsShort.select('b')
  green = maskedImageSelectedBandsShort.select('g')
  red = maskedImageSelectedBandsShort.select('r')
  nir = maskedImageSelectedBandsShort.select('nir')
  swir1 = maskedImageSelectedBandsShort.select('swir1')
  swir2 = maskedImageSelectedBandsShort.select('swir2')

  secondaryMask = ee.Image(blue.gt(3000).And(green.gt(3000)).And(red.gt(3000)).And(nir.gt(3000)).And(swir1.gt(3000)).And(swir2.gt(3000))).neq(1)

  blueVal = ee.Image(blue.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()
  greenVal = ee.Image(green.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()
  redVal = ee.Image(red.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()

  whitenessIndex = ee.Image(blueVal.add(greenVal).add(redVal)).divide(ee.Image(blue.add(green).add(red)).divide(3))
  whitenessMask = whitenessIndex.gt(0.2)
  medianCompWhiteMask = maskedImageSelectedBandsShort.updateMask(whitenessMask)
  medianSecondaryMask = medianCompWhiteMask.updateMask(secondaryMask)

  return medianSecondaryMask

def cloudMaskLSTM(image):
  cloudShadowBitMask = (1 << 4);
  cloudsBitMask = (1 << 3);
  cirrusBitMask = (1<<2)
  snowBitMask = (1<<5)
  qa = image.select('QA_PIXEL');
  ra = image.select('QA_RADSAT')
  mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0)).And(qa.bitwiseAnd(cirrusBitMask).eq(0)).Or(ra.bitwiseAnd(1<<1))
  anySaturated = bitwiseExtract(ra, 1, 7).eq(0)
  snowMask = qa.bitwiseAnd(snowBitMask).eq(0)


  opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2);
  thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0);
  correctedImg =  image.addBands(opticalBands, None, True).addBands(thermalBands, None, True);

  bandNames = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'];
  maskedImage = correctedImg.updateMask(mask);
  maskedImgSnow = maskedImage.updateMask(snowMask)
  maskedImgSnowSat = maskedImgSnow.updateMask(anySaturated)
  maskedImageSelectedBands = maskedImgSnow.select(bandNames);
  maskedImageSelectedBands = maskedImageSelectedBands.rename('b','g','r','nir','swir1','swir2')

  maskedImageSelectedBandsShort = maskedImageSelectedBands.multiply(10000)

  blue = maskedImageSelectedBandsShort.select('b')
  green = maskedImageSelectedBandsShort.select('g')
  red = maskedImageSelectedBandsShort.select('r')
  nir = maskedImageSelectedBandsShort.select('nir')
  swir1 = maskedImageSelectedBandsShort.select('swir1')
  swir2 = maskedImageSelectedBandsShort.select('swir2')

  secondaryMask = ee.Image(blue.gt(3000).And(green.gt(3000)).And(red.gt(3000)).And(nir.gt(3000)).And(swir1.gt(3000)).And(swir2.gt(3000))).neq(1)

  blueVal = ee.Image(blue.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()
  greenVal = ee.Image(green.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()
  redVal = ee.Image(red.subtract(ee.Image(ee.Image(blue.add(green).add(red)).divide(3)))).abs()

  whitenessIndex = ee.Image(blueVal.add(greenVal).add(redVal)).divide(ee.Image(blue.add(green).add(red)).divide(3))
  whitenessMask = whitenessIndex.gt(0.2)
  medianCompWhiteMask = maskedImageSelectedBandsShort.updateMask(whitenessMask)
  medianSecondaryMask = medianCompWhiteMask.updateMask(secondaryMask)

  return medianSecondaryMask

def getLandsatComposite(month,year,geometry):
  #print(geometry.getInfo())
  first_month = month
  last_month = month+1

  first_year = year
  last_year = year

  parScale = 16
  tileScale = 16
  resolution = 30

  landsat9Collection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")\
                      .filterBounds(geometry)\
                      .filter(ee.Filter.calendarRange(first_year,first_year,'year'))\
                      .filter(ee.Filter.calendarRange(first_month,last_month,'month'))
  ls9CloudMasked = landsat9Collection.map(cloudMaskLSOLI)

  landsat8Collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")\
                      .filterBounds(geometry)\
                      .filter(ee.Filter.calendarRange(first_year,first_year,'year'))\
                      .filter(ee.Filter.calendarRange(first_month,last_month,'month'))
  ls8CloudMasked = landsat8Collection.map(cloudMaskLSOLI)

  landsat7Collection = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")\
                      .filterBounds(geometry)\
                      .filter(ee.Filter.calendarRange(first_year,first_year,'year'))\
                      .filter(ee.Filter.calendarRange(first_month,last_month,'month'))
  ls7CloudMasked = landsat7Collection.map(cloudMaskLSTM)

  landsat5Collection = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")\
                      .filterBounds(geometry)\
                      .filter(ee.Filter.calendarRange(first_year,first_year,'year'))\
                      .filter(ee.Filter.calendarRange(first_month,last_month,'month'))
  ls5CloudMasked = landsat5Collection.map(cloudMaskLSTM)

  median_compositeCol = ls8CloudMasked.merge(ls9CloudMasked)

  median_composite = median_compositeCol.reduce(ee.Reducer.median(),parScale).clip(geometry)

  median_composite = median_composite.select(['b_median', 'g_median', 'r_median','nir_median','swir1_median','swir2_median'],['b', 'g', 'r','nir','swir1','swir2'])

  waterMean = ee.List([328.828003, 457.0209459, 388.7936937, 401.134009, 211.4467718, 156.7556306])
  vegMean = ee.List([252.6814193,423.5621885,316.6515439,2950.398944,1365.085618,613.4200433])
  bareMean = ee.List([844.9641061,1286.110196,1564.389302,2703.353268,3411.556313,2502.351313])
  burnMean = ee.List([307.7541667,441.45,498.4208333,807.5208333,1049.75,938.8333333])

  endmembers = ee.List([waterMean,vegMean,bareMean,burnMean])

  unmixedImage = median_composite.unmix(endmembers,True,True);
  bandNames2 = ['water', 'veg', 'bare','burn'];
  unmixedImage = unmixedImage.rename(bandNames2);
  unmixedImage = unmixedImage.select(bandNames2);
  unmixedImage = unmixedImage.multiply(10000)

  # def calcNDMI(img):
  #   ndmi = img.normalizedDifference(['g','swir1'])
  #   return ndmi

  # ndmiCol = median_compositeCol.map(calcNDMI)

  # medianNDMI = ndmiCol.median()



  return unmixedImage

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


In [ ]:
sahelTiles = ee.FeatureCollection('projects/ee-gregoryoakessentone/assets/Sahel_Grid-5_V2_Union')
#sahelTiles = ee.FeatureCollection('projects/ee-gregoryoakessentone/assets/Ethiopia_Grid_2-5_Clip')
#sahelTiles = ee.FeatureCollection('projects/ee-gregoryoakessentone/assets/Kenya_Grid_2-5_Clip')
#sahelTiles = ee.FeatureCollection('projects/ee-gregoryoakessentone/assets/Sudd_Point_Airstrip_PolyBox')
#sahelTiles = ee.FeatureCollection('projects/ee-gregoryoakessentone/assets/Class_Wetland_Rand_Points_PolyBox_V1') ## CD-WA

tileList = sahelTiles.toList(sahelTiles.size())

numTiles = sahelTiles.size().getInfo()

for tileNum in range(0,numTiles):
  # print(i)
  geometry = ee.Feature(tileList.get(tileNum)).geometry()
  year = 1997
  listImgs = []
  # monthList = [1,3,5,7,9,11]
  monthList = [1,3,5,7,9,11]
  for i in monthList:
    listImgs.append(getLandsatComposite(i,year,geometry))

  unMixCol = ee.ImageCollection(listImgs)
  outputImage = unMixCol.toBands()
  task = ee.batch.Export.image.toDrive(
      image=outputImage.short(),
      description='UnmixImage_Bi-Monthly-Tile-{0}_Year-{1}'.format(tileNum,year),
      #description='MNDWI_Bi-Monthly-Tile-{0}_Year-{1}'.format(tileNum,year),
      region=geometry,
      scale=30,
      maxPixels = 1e12,
      folder = 'Sahel_Tiles'
  )
  task.start()